In [17]:
import os, sys
import numpy as np
import pandas as pd
from Bio import SeqIO
from scipy.version import full_version as spversion
from Bio import __version__ as bioversion
import numpy as np
import pandas as pd


print("Python:    ", sys.version)
print("Numpy:     ", np.__version__)
print("Scipy:     ", spversion)
print("Biopython: ", bioversion)
print("Workdir:   ", os.getcwd())

Python:     3.11.16 | packaged by conda-forge | (main, Aug 21 2026, 22:58:50) [Clang 19.1.7 ]
Numpy:      2.4.6
Scipy:      1.17.1
Biopython:  1.88
Workdir:    /Users/annaklimova


In [18]:
PROTEIN_FASTA = (
    "~/ncbi_dataset/"
    "data/GCF_000001635.26/protein.faa"
)

PATH_TO_SAVE_DATA = "~/Hussein/"

ALLELE_LENGTHS = {
    "H2-Kb": 8,
    "H2-Db": 9,
    "I-Ab": 15,
}

N_DECOYS = 100_000
STANDARD_AMINO_ACIDS = set("ACDEFGHIKLMNPQRSTVWY")

In [19]:
proteins = []

for record in SeqIO.parse(PROTEIN_FASTA, "fasta"):
    sequence = str(record.seq).rstrip("*").upper()

    proteins.append(
        {
            "protein_id": record.id,
            "sequence": sequence,
        }
    )

print(f"Loaded proteins: {len(proteins)}")

Loaded proteins: 84,985


In [20]:
def generate_random_decoys(
    proteins,
    peptide_length,
    number_of_peptides=100000,
    seed=42,
):
    rng = np.random.default_rng(seed)

    suitable_proteins = [
        protein
        for protein in proteins
        if len(protein["sequence"]) >= peptide_length
    ]
    window_counts = np.array(
        [
            len(protein["sequence"]) - peptide_length + 1
            for protein in suitable_proteins
        ],
        dtype=np.int64,
    )

    cumulative_counts = np.cumsum(window_counts)
    total_windows = int(cumulative_counts[-1])

    selected = {}

    while len(selected) < number_of_peptides:
        remaining = number_of_peptides - len(selected)


        batch_size = max(10_000, remaining * 2)

        global_positions = rng.integers(
            0,
            total_windows,
            size=batch_size,
        )

        protein_indices = np.searchsorted(
            cumulative_counts,
            global_positions,
            side="right",
        )

        for global_position, protein_index in zip(
            global_positions,
            protein_indices,
        ):
            previous_total = (
                0
                if protein_index == 0
                else cumulative_counts[protein_index - 1]
            )

            peptide_start = int(global_position - previous_total)

            protein = suitable_proteins[protein_index]
            sequence = protein["sequence"]

            peptide = sequence[
                peptide_start:
                peptide_start + peptide_length
            ]

            if not set(peptide).issubset(STANDARD_AMINO_ACIDS):
                continue

            if peptide not in selected:
                selected[peptide] = {
                    "peptide": peptide,
                    "protein_id": protein["protein_id"],
                    "peptide_start": peptide_start + 1,
                    "peptide_end": peptide_start + peptide_length,
                }

            if len(selected) == number_of_peptides:
                break

    return pd.DataFrame(selected.values())

In [21]:
os.makedirs(PATH_TO_SAVE_DATA, exist_ok=True)

decoy_tables = {}

for index, (allele, peptide_length) in enumerate(
    ALLELE_LENGTHS.items()
):
    print(
        f"Generating {N_DECOYS:,} decoys "
        f"for {allele}, length {peptide_length}"
    )

    decoy_df = generate_random_decoys(
        proteins=proteins,
        peptide_length=peptide_length,
        number_of_peptides=N_DECOYS,
        seed=42 + index,
    )

    decoy_df.insert(0, "allele", allele)
    decoy_df.insert(1, "length", peptide_length)

    safe_allele = allele.replace("*", "").replace(":", "-")

    tsv_path = os.path.join(
        PATH_TO_SAVE_DATA,
        f"decoy-mouse-allele-{safe_allele}"
        f"-length-{peptide_length}.tsv",
    )

    txt_path = os.path.join(
        PATH_TO_SAVE_DATA,
        f"decoy-mouse-allele-{safe_allele}"
        f"-length-{peptide_length}.txt",
    )

    decoy_df.to_csv(
        tsv_path,
        sep="\t",
        index=False,
    )

    decoy_df["peptide"].to_csv(
        txt_path,
        index=False,
        header=False,
    )

    decoy_tables[allele] = decoy_df

    print(f"Saved TSV: {tsv_path}")
    print(f"Saved TXT: {txt_path}")

Generating 100,000 decoys for H2-Kb, length 8
Saved TSV: /Users/annaklimova/Hussein/decoy-mouse-allele-H2-Kb-length-8.tsv
Saved TXT: /Users/annaklimova/Hussein/decoy-mouse-allele-H2-Kb-length-8.txt
Generating 100,000 decoys for H2-Db, length 9
Saved TSV: /Users/annaklimova/Hussein/decoy-mouse-allele-H2-Db-length-9.tsv
Saved TXT: /Users/annaklimova/Hussein/decoy-mouse-allele-H2-Db-length-9.txt
Generating 100,000 decoys for I-Ab, length 15
Saved TSV: /Users/annaklimova/Hussein/decoy-mouse-allele-I-Ab-length-15.tsv
Saved TXT: /Users/annaklimova/Hussein/decoy-mouse-allele-I-Ab-length-15.txt
